<a href="https://colab.research.google.com/github/diegopadilla-esfm/ModelosEstocasticosESFM/blob/main/UniformizacionCMTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Resultados

Se calcularon las matrices $P(0.5)$, $P(1)$ y $P(5)$ utilizando

$$
P(t)=\sum_{k=0}^{M} e^{-rt}\frac{(rt)^k}{k!}\hat P^k.
$$

con

$$
M=\left\lceil \max\{rt+5\sqrt{rt},20\}\right\rceil.
$$

Posteriormente se verificó la ecuación de Chapman--Kolmogorov calculando

$$
P(0.5)^2
$$

y comparándola con

$$
P(1).
$$

El error máximo obtenido fue del orden del error numérico de redondeo, por lo que la ecuación de Chapman--Kolmogorov se verifica numéricamente.

In [11]:
import numpy as np
from scipy.special import factorial

# Matriz de tasas
R = np.array([
    [0,2,3,0],
    [4,0,2,0],
    [0,2,0,2],
    [1,0,3,0]
],dtype=float)

r = 6

# tasas de salida
ri = np.sum(R,axis=1)

# P sombrero
Phat = np.zeros((4,4))

for i in range(4):
    for j in range(4):

        if i == j:
            Phat[i,j] = 1-ri[i]/r

        else:
            Phat[i,j] = R[i,j]/r

print("P sombrero:")
print(Phat)

def uniformizacion(t):

    M = int(np.ceil(max(r*t + 5*np.sqrt(r*t),20)))

    P = np.zeros((4,4))

    potencia = np.eye(4)

    for k in range(M+1):

        coef = np.exp(-r*t)*(r*t)**k/factorial(k)

        P += coef*potencia

        potencia = potencia @ Phat

    return P,M

for t in [0.5,1,5]:

    P,M = uniformizacion(t)

    print("\n====================")
    print("t =",t)
    print("M =",M)
    print(P)

P sombrero:
[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]

t = 0.5
M = 20
[[0.25060868 0.2169646  0.38665694 0.14576979]
 [0.25313484 0.23836098 0.37440924 0.13409493]
 [0.1691195  0.19361489 0.42030102 0.2169646 ]
 [0.15801748 0.15744464 0.39833179 0.28620609]]

t = 1
M = 20
[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]

t = 5
M = 58
[[0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999962 0.19999962 0.39999925 0.19999963]
 [0.19999962 0.19999962 0.39999925 0.19999963]]


In [12]:
P05,_ = uniformizacion(0.5)

P1,_ = uniformizacion(1)

CK = P05 @ P05

print("\nP(1)")
print(P1)

print("\nP(0.5)^2")
print(CK)

print("\nError máximo:")
print(np.max(np.abs(P1-CK)))


P(1)
[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]

P(0.5)^2
[[0.20615141 0.20390232 0.39871018 0.19123609]
 [0.20828451 0.20534099 0.39789976 0.18847475]
 [0.19675878 0.19837963 0.40095927 0.20390232]
 [0.19204651 0.19399744 0.40147152 0.21248452]]

Error máximo:
5.820333639494635e-07


## Algoritmo de uniformización con tolerancia

Se aplicó el algoritmo de uniformización con tolerancia

$$
\varepsilon = 10^{-5}.
$$

Para cada valor de $t$ se obtuvo automáticamente un valor de $M$
suficiente para garantizar que

$$
\sum_{k=M+1}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\leq \varepsilon.
$$

Las matrices obtenidas se compararon con las calculadas en el ejercicio
anterior utilizando

$$
M \approx \max\{rt+5\sqrt{rt},\,20\}.
$$

En todos los casos las diferencias fueron pequeñas, confirmando que la
aproximación propuesta produce resultados muy cercanos a los obtenidos
mediante el criterio de tolerancia.

In [13]:
import numpy as np
from scipy.stats import poisson

def uniformizacion_epsilon(t, eps=1e-5):

    n = len(Phat)

    A = Phat.copy()

    B = np.exp(-r*t) * np.eye(n)

    c = np.exp(-r*t)

    suma = c

    k = 1

    while suma < 1 - eps:

        c = c * (r*t) / k

        B = B + c*A

        A = A @ Phat

        suma += c

        k += 1

    M = k - 1


    error_teorico = 1 - poisson.cdf(M, r*t)

    return B, M, error_teorico


for t in [0.5, 1, 5]:


    P_aprox, M_aprox = uniformizacion(t)

    # Método del ejercicio 4
    P_eps, M, error_teorico = uniformizacion_epsilon(t)

    # Error observado entre ambos métodos
    error_real = np.max(np.abs(P_aprox - P_eps))

    print("\n====================")
    print("t =", t)

    print("\n ejercicio 3")
    print("M =", M_aprox)

    print("\n ejercicio 4")
    print("M =", M)

    print("\nError teórico <=", error_teorico)

    print("Error real =", error_real)

    print("\nP(t) con tolerancia:")
    print(P_eps)


t = 0.5

 ejercicio 3
M = 20

 ejercicio 4
M = 13

Error teórico <= 3.4019146132324707e-06
Error real = 1.3607613894017767e-06

P(t) con tolerancia:
[[0.250608   0.21696392 0.38665557 0.14576911]
 [0.25313416 0.2383603  0.37440788 0.13409425]
 [0.16911882 0.19361421 0.42029966 0.21696392]
 [0.1580168  0.15744396 0.39833043 0.28620541]]

t = 1

 ejercicio 3
M = 20

 ejercicio 4
M = 19

Error teórico <= 5.180168937024554e-06
Error real = 1.4900247798932398e-06

P(t) con tolerancia:
[[0.20615038 0.20390128 0.39870811 0.19123506]
 [0.20828347 0.20533995 0.39789768 0.18847371]
 [0.19675775 0.19837859 0.4009572  0.20390128]
 [0.19204548 0.1939964  0.40146945 0.21248349]]

t = 5

 ejercicio 3
M = 58

 ejercicio 4
M = 56

Error teórico <= 7.378955003023435e-06
Error real = 2.2001289619599795e-06

P(t) con tolerancia:
[[0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999852 0.19999852 0.39999705 0.19999853]
 [0.19999852 0.19999852 0.39999705 0.